# 05 · Outcome analysis
**RQ:** does a team whose players were more role-consistent in *prior* rounds
win the *current* round more often?

Main model: round-level logit, `t_win ~ cons_diff + equip_diff + score_diff +
round_num + pistol + map FE`, SEs clustered by match. Interpreted as a
**correlation**, per the research question.

In [ ]:
# --- bootstrap (identical in every notebook) ---------------------------------
from google.colab import drive
drive.mount('/content/drive')
%pip -q install demoparser2 minisom

import sys
sys.path.insert(0, '/content/drive/MyDrive/cs2-btp/code')

from cs2btp import (config as cfg, manifest as mf, parsing, qc,
                    features as ft, roles, consistency as cons,
                    outcome as out, viz)
cfg.ensure_dirs()
print('pipeline ready | drive root =', cfg.DRIVE_ROOT)

In [ ]:
import pandas as pd
rolled = pd.read_parquet(cfg.ANALYSIS/'consistency_rolling.parquet')
team_round = pd.read_parquet(cfg.ANALYSIS/'team_round_consistency.parquet')
ds = out.build_round_dataset(team_round)
print('rounds in dataset:', len(ds), '| usable:', ds.cons_diff.notna().sum())

In [ ]:
model, d = out.fit_main_logit(ds)
print(model.summary())

In [ ]:
viz.coef_plot(model)
qt = out.quartile_table(ds)
print(qt.to_string(index=False))
viz.quartile_bar(qt);

## Alternative consistency definitions (should point the same way)

In [ ]:
for var in ['cons_min_diff', 'entropy_diff']:
    m, dd = out.fit_main_logit(ds, cons_var=var)
    print(f"{var:15s} beta = {m.params[var]: .3f}  p = {m.pvalues[var]:.4f}  n = {len(dd)}")

## Coarser half-level check

In [ ]:
labeled = roles.load_labeled()
mlv = pd.read_parquet(cfg.ANALYSIS/'consistency_match_level.parquet')
m2, dhalf = out.half_level_ols(mlv, labeled)
print(m2.summary())

### Interpretation guardrails (write these into the thesis)
* Rolling consistency uses **only prior rounds** → no mechanical leakage.
* Still a **correlation**: winning teams get to keep executing their plan,
  losers are forced to improvise. Discuss this reverse-pressure channel openly.
* Effective sample for the consistency signal ≈ number of matches, not rounds
  (it varies mostly between teams) — with ~25 series expect wide CIs.